EDA - wczytanie danych i analiza

In [55]:
import pandas as pd
import numpy as np

df = pd.read_json('fast_cars_data.json')

df.info()
df.describe()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 15968 entries, 0 to 15967
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Tytuł              15968 non-null  str  
 1   Cena               15968 non-null  str  
 2   URL                15968 non-null  str  
 3   Rok produkcji      15968 non-null  int64
 4   Przebieg           15968 non-null  str  
 5   Pojemność skokowa  15968 non-null  str  
 6   Moc                15968 non-null  str  
 7   Rodzaj paliwa      15968 non-null  str  
 8   Skrzynia biegów    15968 non-null  str  
dtypes: int64(1), str(8)
memory usage: 1.1 MB


,Tytuł,Cena,URL,Rok produkcji,Przebieg,Pojemność skokowa,Moc,Rodzaj paliwa,Skrzynia biegów
0,Mercedes-Benz EQE 350 89kWh 4-Matic Electric Art,327809 PLN,https://www.otomoto.pl/osobowe/oferta/mercedes...,2025,1 km,Brak,292 KM,Elektryczny,Automatyczna
1,Volvo S90 T5 Geartronic Inscription,59000 PLN,https://www.otomoto.pl/osobowe/oferta/volvo-s9...,2020,118760 km,1969 cm3,254 KM,Benzyna,Automatyczna
2,BMW Seria 5 530i M Sport Edition,80000 PLN,https://www.otomoto.pl/osobowe/oferta/bmw-seri...,2019,120000 km,1998 cm3,252 KM,Benzyna,Automatyczna
3,Tesla Model S Plaid,239000 PLN,https://www.otomoto.pl/osobowe/oferta/tesla-mo...,2022,22 km,Brak,1020 KM,Elektryczny,Automatyczna
4,BAIC 3,74900 PLN,https://www.otomoto.pl/osobowe/oferta/baic-3-I...,2026,5 km,1499 cm3,136 KM,Benzyna,Manualna


In [56]:
df = df.replace('Brak', np.nan)

braki = df[['Rok produkcji', 'Cena', 'Przebieg', 'Pojemność skokowa', 'Moc', 'Rodzaj paliwa', 'Skrzynia biegów']].isna().sum()
print(braki)

warunek_brak_pojemnosci = df['Pojemność skokowa'].isna()
warunek_elektryk = df['Rodzaj paliwa'] == 'Elektryczny'
warunek_moc = df['Moc'].isna()

unikalne_paliwa = df['Rodzaj paliwa'].unique()

print("Dostępne rodzaje paliwa w bazie:")
print(unikalne_paliwa)

for typ_paliwa in unikalne_paliwa:
    warunek_paliwo = df['Rodzaj paliwa'] == typ_paliwa
    print(f"Liczba aut o paliwie \"{typ_paliwa}\": {warunek_paliwo.sum()}")
    print(f"Ilość wpisów z brakującą pojemnością skokową dla tego paliwa \"{typ_paliwa}\": {(warunek_paliwo & warunek_brak_pojemnosci).sum()}")
    print(f"Ilość wpisów z brakującą mocą dla tego paliwa \"{typ_paliwa}\": {(warunek_paliwo & warunek_moc).sum()}")


Rok produkcji          0
Cena                   0
Przebieg               0
Pojemność skokowa    312
Moc                  126
Rodzaj paliwa          0
Skrzynia biegów        1
dtype: int64
Dostępne rodzaje paliwa w bazie:
<StringArray>
[    'Elektryczny',         'Benzyna', 'Hybryda Plug-in',          'Diesel',
         'Hybryda',     'Benzyna+LPG',     'Benzyna+CNG',           'Wodór']
Length: 8, dtype: str
Liczba aut o paliwie "Elektryczny": 256
Ilość wpisów z brakującą pojemnością skokową dla tego paliwa "Elektryczny": 256
Ilość wpisów z brakującą mocą dla tego paliwa "Elektryczny": 2
Liczba aut o paliwie "Benzyna": 7944
Ilość wpisów z brakującą pojemnością skokową dla tego paliwa "Benzyna": 27
Ilość wpisów z brakującą mocą dla tego paliwa "Benzyna": 65
Liczba aut o paliwie "Hybryda Plug-in": 434
Ilość wpisów z brakującą pojemnością skokową dla tego paliwa "Hybryda Plug-in": 2
Ilość wpisów z brakującą mocą dla tego paliwa "Hybryda Plug-in": 3
Liczba aut o paliwie "Diesel": 5959
Ilość

Czyszczenie danych i zamiana tekstu na liczby

In [57]:
print(f"Rozmiar bazy na start: {len(df)}")

df = df[~df['Rodzaj paliwa'].isin(['Benzyna+CNG', 'Wodór'])]

df.loc[df['Rodzaj paliwa'] == 'Elektryczny', 'Pojemność skokowa'] = '0'

df = df.dropna(subset=['Moc', 'Pojemność skokowa', 'Skrzynia biegów'])

print(f"Rozmiar bazy po usunięciu braków i mikrogrup: {len(df)}")


df['Cena'] = df['Cena'].astype(str)
df = df[df['Cena'].str.contains('PLN', na=False)]
df['Cena'] = df['Cena'].str.replace(' PLN', '', regex=False)
df['Cena'] = df['Cena'].str.replace(' ', '', regex=False)
df['Cena'] = df['Cena'].astype(float).astype(int)

df['Przebieg'] = df['Przebieg'].astype(str).str.replace(' km', '', regex=False)
df['Przebieg'] = df['Przebieg'].str.replace(' ', '', regex=False)
df['Przebieg'] = df['Przebieg'].astype(int)

df['Moc'] = df['Moc'].astype(str).str.replace(' KM', '', regex=False)
df['Moc'] = df['Moc'].str.replace(' ', '', regex=False)
df['Moc'] = df['Moc'].astype(int)

df['Pojemność skokowa'] = df['Pojemność skokowa'].astype(str).str.replace(' cm3', '', regex=False)
df['Pojemność skokowa'] = df['Pojemność skokowa'].str.replace(' ', '', regex=False)
# Tu używamy float a potem int, żeby ładnie przekonwertować zera
df['Pojemność skokowa'] = df['Pojemność skokowa'].astype(float).astype(int) 

print("\nSukces! Twoje dane są teraz czystymi liczbami.")

Rozmiar bazy na start: 15968
Rozmiar bazy po usunięciu braków i mikrogrup: 15810

Sukces! Twoje dane są teraz czystymi liczbami.


In [58]:
df.info()
df.describe()

<class 'pandas.DataFrame'>
Index: 15781 entries, 0 to 15967
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Tytuł              15781 non-null  str  
 1   Cena               15781 non-null  int64
 2   URL                15781 non-null  str  
 3   Rok produkcji      15781 non-null  int64
 4   Przebieg           15781 non-null  int64
 5   Pojemność skokowa  15781 non-null  int64
 6   Moc                15781 non-null  int64
 7   Rodzaj paliwa      15781 non-null  str  
 8   Skrzynia biegów    15781 non-null  str  
dtypes: int64(5), str(4)
memory usage: 1.2 MB


,Cena,Rok produkcji,Przebieg,Pojemność skokowa,Moc
count,1.578100e+04,15781.000000,1.578100e+04,15781.000000,15781.000000
mean,8.531421e+04,2016.370699,1.379145e+05,1870.829352,173.549078
std,1.156902e+05,7.298338,1.023111e+05,786.085933,93.456660
min,1.000000e+03,1956.000000,0.000000e+00,0.000000,1.000000
25%,2.450000e+04,2012.000000,5.100000e+04,1490.000000,116.000000
50%,4.999900e+04,2017.000000,1.380000e+05,1798.000000,150.000000
75%,1.079000e+05,2022.000000,2.035000e+05,1996.000000,197.000000
max,4.522000e+06,2026.000000,2.861515e+06,6749.000000,1401.000000
